In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
#from Database.TPData import TPData, TPDataDa, TPDataAssembly

from Strategies.LeadLagXGB.backtest_class import BacktestLL
from Strategies.LeadLagXGB.strategy_class import StrategyLL, VolumeClass
tol=(1e-1)/2

from bokeh.plotting import figure, show, output_file
from bokeh.models import ColumnDataSource, DatetimeTickFormatter, HoverTool
from bokeh.layouts import column
from bokeh.io import curdoc

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import warnings
warnings.filterwarnings('ignore')

# OOT part - Nov, Dec

In [4]:
data_lag = pd.read_csv(r's:\Algo\Files\andrej\Data\int_data_lag_production_sample_dem2_our_db.csv',
                    parse_dates=['datetime']).reset_index()

In [5]:
data_lag['time_diff']=data_lag['datetime'].diff().dt.total_seconds().fillna(0)

In [6]:
data_lag=data_lag[data_lag['datetime'].apply(lambda x: x.hour>8 and  x.hour<18)]

In [7]:
#data_lag=data_lag.drop(index=177223).reset_index()
# del data_lag['level_0']
# del data_lead['level_0']

In [8]:
data_lag.head()

,index,datetime,trd_price,volume,bid_price,ask_price,mid_price,trd_side,time_diff
0,0,2025-02-28 09:00:03.401,NaN,NaN,75.11,75.49,75.300,NaN,0.000
1,1,2025-02-28 09:00:05.901,NaN,NaN,74.82,75.49,75.155,NaN,2.500
2,2,2025-02-28 09:00:05.944,NaN,NaN,75.03,75.49,75.260,NaN,0.043
3,3,2025-02-28 09:00:06.027,NaN,NaN,75.09,75.49,75.290,NaN,0.083
4,4,2025-02-28 09:00:09.401,NaN,NaN,74.84,75.49,75.165,NaN,3.374


## BA_convergence analysis

In [9]:
data_lag['ba_spread']=data_lag['ask_price']-data_lag['bid_price']

# Constructing BA convergence predictor

In [10]:
df=data_lag[data_lag['trd_price'].isnull()==True]

In [11]:
df

,index,datetime,trd_price,volume,bid_price,ask_price,mid_price,trd_side,time_diff,ba_spread
0,0,2025-02-28 09:00:03.401,NaN,NaN,75.11,75.49,75.300,NaN,0.000000,0.38
1,1,2025-02-28 09:00:05.901,NaN,NaN,74.82,75.49,75.155,NaN,2.500000,0.67
2,2,2025-02-28 09:00:05.944,NaN,NaN,75.03,75.49,75.260,NaN,0.043000,0.46
3,3,2025-02-28 09:00:06.027,NaN,NaN,75.09,75.49,75.290,NaN,0.083000,0.40
4,4,2025-02-28 09:00:09.401,NaN,NaN,74.84,75.49,75.165,NaN,3.374000,0.65
...,...,...,...,...,...,...,...,...,...,...
793168,793168,2025-05-06 17:59:40.932,NaN,NaN,77.47,77.60,77.535,NaN,0.704000,0.13
793169,793169,2025-05-06 17:59:41.567,NaN,NaN,77.42,77.60,77.510,NaN,0.635000,0.18
793170,793170,2025-05-06 17:59:41.808,NaN,NaN,77.43,77.60,77.515,NaN,0.241000,0.17
793172,793172,2025-05-06 17:59:56.008,NaN,NaN,77.42,77.60,77.510,NaN,7.545059,0.18


In [12]:
# Parsing datetime column
df['datetime'] = pd.to_datetime(df['datetime'])

# Sorting data by datetime to ensure proper order
df.sort_values(by='datetime', inplace=True)

# Setting datetime as index
df.set_index('datetime', inplace=True)

# Efficient MACD calculation without row-by-row operations
def calculate_rolling_macd_fast(df, column, short_window=10, long_window=22):
    
    """
    Pre-compute MACD for the entire dataset.
    """
    # Calculate short and long EMA
    short_ema = df[column].ewm(span=short_window, adjust=False).mean()
    long_ema = df[column].ewm(span=long_window, adjust=False).mean()
    macd = short_ema - long_ema
    
    volatility = df[column].rolling(window=long_window).max() - df[column].rolling(window=long_window).min()
    return macd,volatility

# Step 1: Precompute MACD for bid_price and ask_price
df['bid_macd'], df['bid_volatility'] = calculate_rolling_macd_fast(df, 'bid_price')
df['ask_macd'], df['ask_volatility'] = calculate_rolling_macd_fast(df, 'ask_price')

# Step 2: Retain only rows with valid data for a 5-minute lookback
def filter_by_lookback(df, time_col, lookback_minutes=30):
    """
    Keep rows within a specific lookback window for every timestamp.
    """
    lookback = pd.Timedelta(minutes=lookback_minutes)
    df['valid_macd'] = df.index.to_series().diff().le(lookback)
    return df

# Apply the lookback filter
df = filter_by_lookback(df, 'datetime', lookback_minutes=5)

# Drop intermediate 'valid_macd' column for clarity
df.drop(columns=['valid_macd'], inplace=True)

df=df.reset_index()

In [13]:
# Filter data for a specific date (e.g., '2024-11-11')
selected_date = "2024-3-11"
df['datetime'] = pd.to_datetime(df['datetime'])
df_filtered = df[df['datetime'].dt.date == pd.to_datetime(selected_date).date()]

# Convert to Bokeh ColumnDataSource for interaction
source = ColumnDataSource(df_filtered)

# Output file
output_file("bid_ask_spread_hover.html")

# Create bid and ask price plot
p1 = figure(
    x_axis_type="datetime",
    title="Bid and Ask Prices",
    sizing_mode="stretch_both"
)
p1.line(
    x='datetime', y='bid_price', source=source,
    legend_label="Bid Price", line_width=2, color="blue", alpha=0.7
)
p1.line(
    x='datetime', y='ask_price', source=source,
    legend_label="Ask Price", line_width=2, color="green", alpha=0.7
)
p1.add_tools(HoverTool(
    tooltips=[("Datetime", "@datetime{%F %T}"), ("Bid Price", "@bid_price"), ("Ask Price", "@ask_price")],
    formatters={'@datetime': 'datetime'},
    mode='vline'
))
p1.legend.location = "top_left"
p1.yaxis.axis_label = "Price"
p1.xaxis.formatter = DatetimeTickFormatter(hourmin="%H:%M")

# Create bid-ask spread plot
p2 = figure(
    x_axis_type="datetime",
    title="Bid-Ask Spread",
    sizing_mode="stretch_both",
    x_range=p1.x_range
)
p2.line(
    x='datetime', y='ba_spread', source=source,
    line_width=2, color="purple", alpha=0.7
)
p2.add_tools(HoverTool(
    tooltips=[("Datetime", "@datetime{%F %T}"), ("Bid-Ask Spread", "@ba_spread")],
    formatters={'@datetime': 'datetime'},
    mode='vline'
))
p2.yaxis.axis_label = "Spread"
p2.xaxis.formatter = DatetimeTickFormatter(hourmin="%H:%M")

# Calculate percentiles
percentiles = {
    "10th": np.percentile(df_filtered['ba_spread'].dropna(), 10),
    "50th": np.percentile(df_filtered['ba_spread'].dropna(), 50),
    "90th": np.percentile(df_filtered['ba_spread'].dropna(), 90),
}

# Add percentile lines to the second plot
p2.line(
    x=df_filtered['datetime'], y=[percentiles["10th"]] * len(df_filtered),
    line_width=1, color="orange", alpha=0.7, legend_label="10th Percentile"
)
p2.line(
    x=df_filtered['datetime'], y=[percentiles["50th"]] * len(df_filtered),
    line_width=1, color="red", alpha=0.7, legend_label="50th Percentile (Median)"
)
p2.line(
    x=df_filtered['datetime'], y=[percentiles["90th"]] * len(df_filtered),
    line_width=1, color="green", alpha=0.7, legend_label="90th Percentile"
)

# Add hover tool for percentile lines
p2.add_tools(HoverTool(
    tooltips=[
        ("Datetime", "@datetime{%F %T}"),
        ("Bid-Ask Spread", "@ba_spread"),
        ("10th Percentile", str(percentiles["10th"])),
        ("50th Percentile", str(percentiles["50th"])),
        ("90th Percentile", str(percentiles["90th"]))
    ],
    formatters={'@datetime': 'datetime'},
    mode='vline'
))

# Highlight points based on the condition
highlighted_points = df_filtered[
    ((df_filtered['bid_macd'].abs() > 0.04) & (df_filtered['ask_macd'].abs() < 0.01) & (df_filtered['ask_volatility'].abs() < 0.04) |
     (df_filtered['ask_macd'].abs() > 0.04) & (df_filtered['bid_macd'].abs() < 0.01) & (df_filtered['bid_volatility'].abs() < 0.04)) &
    (df_filtered['ba_spread'] < percentiles["10th"])
]

# Create a ColumnDataSource for highlighted points
highlight_source = ColumnDataSource(highlighted_points)

# Add markers to the first plot for highlighted points
p1.scatter(
    x='datetime', y='bid_price', source=highlight_source,
    size=10, color="red", legend_label="Highlighted Points", alpha=0.8
)
p1.scatter(
    x='datetime', y='ask_price', source=highlight_source,
    size=10, color="red", alpha=0.8
)

# Add markers to the second plot for highlighted points
p2.scatter(
    x='datetime', y='ba_spread', source=highlight_source,
    size=10, color="red", legend_label="Highlighted Points", alpha=0.8
)

# Update hover tools for the highlighted points
highlight_hover = HoverTool(
    tooltips=[
        ("Datetime", "@datetime{%F %T}"),
        ("Bid Price", "@bid_price"),
        ("Ask Price", "@ask_price"),
        ("Bid MACD", "@bid_macd"),
        ("Ask MACD", "@ask_macd"),
        ("Bid-Ask Spread", "@ba_spread")
    ],
    formatters={'@datetime': 'datetime'},
    mode='vline'
)

p1.add_tools(highlight_hover)
p2.add_tools(highlight_hover)

# Arrange in a vertical layout and show
layout = column(p1, p2, sizing_mode="stretch_both")
curdoc().add_root(layout)
show(layout)

IndexError: cannot do a non-empty take from an empty axes.

# Backtesting

In [ ]:
from Strategies.BA_convergence.backtest_class import BacktestLL
from Strategies.BA_convergence.strategy_class import StrategyLL, VolumeClass

In [ ]:
#0.04	0.020	0.05	0.09	0.05	0.00	0.5	0.5

In [ ]:
ba_conv_large_threshold=0.04
ba_conv_small_threshold=0.02
ba_conv_volatility_threshold=0.05
ba_conv_ba_threshold=0.09

burnout_period=60
stop_profit = 0.3
makeagg_ratio = 0.6
trail_stop=0.5

take_profit=2
stop_loss=-1

br_fee = 0.0175
closing_mode='Martinovo_zatvaranie'

MACD_long_threshold = 0.05
MACD_short_threshold = -0.05

minimum_intensity=0.5

In [ ]:
instr = 'm2'

contr_vars = []

param_list = ['t_end', 'take_profit', 'stop_loss']
param_dict = {k: [] for k in param_list}
param_dict['t_end'] = time(17)
param_dict['take_profit'] = take_profit
param_dict['stop_loss'] = stop_loss
param_dict['ba_max']=0.3

param_dict['br_fee'] = br_fee


param_dict['ba_conv_large_threshold']=ba_conv_large_threshold
param_dict['ba_conv_small_threshold']=ba_conv_small_threshold
param_dict['ba_conv_volatility_threshold']=ba_conv_volatility_threshold
param_dict['ba_conv_ba_threshold']=ba_conv_ba_threshold

param_dict['MACD_long_threshold'] = MACD_long_threshold
param_dict['MACD_short_threshold'] = MACD_short_threshold

param_dict['minimum_intensity'] = minimum_intensity

param_dict['burnout_period']=burnout_period
param_dict['stop_profit']=stop_profit
param_dict['makeagg_ratio']=makeagg_ratio
param_dict['trail_stop']=trail_stop



vol_class = VolumeClass(1, {'max_clips': 1})
backtest_class = BacktestLL(vol_class)
# model_class = HawkesIntensity
strategy_class = StrategyLL(
                            strategy='LL',
                            market='de',
                            instrument=instr, 
                            is_overnight=False,
                            closing_mode=closing_mode)

strategy_class.load_params(param_dict, contr_vars)

xx = backtest_class.simulate_strategy(strategy_class, instr, data_lag)

df = pd.DataFrame({k: strategy_class.stats_dict[k] for k in list(strategy_class.stats_dict.keys()) if k in ['timestamp', 'position', 'price_level']})
df = pd.concat([df, ])

In [ ]:
from bokeh.plotting import figure, show, output_file
from bokeh.models import ColumnDataSource, Range1d, LinearAxis, HoverTool
import pandas as pd

# As

# Assign color based on 'position' value
df['color'] = df['position'].map({1: 'green', -1: 'red', 0: 'black'})
df['price_level']=df['price_level']-df['position']*br_fee

# Create ColumnDataSources
source_df = ColumnDataSource(df)
source_lag = ColumnDataSource(data_lag)

# Create a figure with dynamic sizing
p = figure(x_axis_type="datetime", title="Strategy Positions", sizing_mode='stretch_both')

# Adjust y-axes ranges
p.extra_y_ranges = {
   "Lag price": Range1d(start=data_lag['trd_price'].min(), end=data_lag['trd_price'].max())
}

# Add left and right y-axes
p.add_layout(LinearAxis(y_range_name="Lag price", axis_label="Lag Price"), 'right')

# Plotting with hover tools
lag_crosses = p.cross(x='datetime', y='trd_price', source=source_lag, size=10, color='orange', legend_label="Lag Price", y_range_name="Lag price")
position_circles = p.circle(x='timestamp', y='price_level', source=source_df, size=10, color='color', legend_label="Position", y_range_name="Lag price")

# Define hover tools for each glyph
hover_position = HoverTool(renderers=[position_circles], tooltips=[
    ("Timestamp", "@timestamp{%F %T}"),
    ("Price Level", "@price_level"),
    ("Position", "@position"),
    ("Color", "@color"),
], formatters={'@timestamp': 'datetime'})


hover_lag = HoverTool(renderers=[lag_crosses], tooltips=[
    ("Datetime", "@datetime{%F %T}"),
    ("Lag Price", "@trd_price"),
], formatters={'@datetime': 'datetime'})

# Add hover tools to the plot
p.add_tools(hover_position, hover_lag)

# Customize plot
p.legend.location = "top_left"
p.xaxis.axis_label = "Datetime"
p.yaxis.axis_label = "Price Level"

# Show the plot
show(p)

In [ ]:
xx.plot()

In [ ]:
xx[-1]

In [ ]:
df['date']=df['timestamp'].apply(lambda x: x.date())

In [ ]:
df['revenue']=-1*df['position']*df['price_level']

In [ ]:
# Select odd and even indexed rows
even_index = df.iloc[1::2]['revenue'].reset_index(drop=True)  # Rows 1, 3, 5, ...
odd_index = df.iloc[0::2]['revenue'].reset_index(drop=True)   # Rows 0, 2, 4, ...

# Subtract odd-indexed rows from even-indexed rows
difference = even_index + odd_index
reutrn_df=pd.DataFrame(difference)
reutrn_df.columns=['returns']
len(reutrn_df)

In [ ]:
a=df.groupby('date')['revenue'].sum()

In [ ]:
(a>0).value_counts()

In [ ]:
# Assuming 'reutrn_df' is a DataFrame and 'returns' is a column in it
overall_mean = reutrn_df['returns'].mean()
positive_mean = reutrn_df[reutrn_df['returns'] > 0]['returns'].mean()
negative_mean = reutrn_df[reutrn_df['returns'] < 0]['returns'].mean()
 
# Print accuracy
accuracy = sum(reutrn_df['returns'] > 0) / len(reutrn_df)
print('Accuracy:', accuracy)
 
# Plot histogram
reutrn_df[abs(reutrn_df['returns']) >= 0.01]['returns'].hist()
 
# Add vertical lines
plt.axvline(overall_mean, color='r', linestyle='dashed', linewidth=2, label=f'Overall Mean: {overall_mean:.2f}')
plt.axvline(positive_mean, color='g', linestyle='dashed', linewidth=2, label=f'Positive Mean: {positive_mean:.2f}')
plt.axvline(negative_mean, color='b', linestyle='dashed', linewidth=2, label=f'Negative Mean: {negative_mean:.2f}')
 
# Add a legend
plt.legend()
 
# Show plot
plt.show()